In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from timm.data import ImageDataset, create_transform, resolve_data_config
from compression.token_pruner import *

# [!] Assume your custom classes (CompressedAttention, CompressedBlock, 
# create_compressed_deit_small, TokenPruner, etc.) are defined above.

def get_imagenet_sample_timm(dataset_root: str, model_config: dict, target_idx: int = 0):
    """
    1. Read a specific sample from ImageNet using timm.data utilities.
    """
    # Create raw dataset to fetch the un-transformed PIL Image for visualization
    dataset_raw = ImageDataset(dataset_root)
    img_pil, label = dataset_raw[target_idx]
    
    # Resolve standard data configuration for the model (e.g., input size, mean, std)
    data_config = resolve_data_config(model_config)
    
    # Create standard validation transforms using timm
    eval_transform = create_transform(**data_config, is_training=False)
    
    # Create visualization transforms (Resize & Crop ONLY, without ToTensor/Normalize)
    # This ensures the visualized image perfectly aligns with the model's spatial input
    input_size = data_config['input_size'][-1] # Usually 224
    crop_pct = data_config.get('crop_pct', 0.875)
    scale_size = int(input_size / crop_pct)
    
    # We manually resize and crop the PIL image to match the tensor's spatial dimensions
    img_vis = img_pil.resize((scale_size, scale_size), Image.BICUBIC)
    left = (scale_size - input_size) / 2
    top = (scale_size - input_size) / 2
    right = (scale_size + input_size) / 2
    bottom = (scale_size + input_size) / 2
    img_vis = img_vis.crop((left, top, right, bottom))
    
    # Apply standard timm evaluation transforms to get the model input tensor
    img_tensor = eval_transform(img_pil).unsqueeze(0)
    
    return img_tensor, img_vis, label

def extract_pruning_metadata(model):
    """
    2. Register forward hooks to capture pruning metadata dynamically.
    (Non-intrusive way to catch the discarded metadata dict in your Block)
    """
    pruned_indices_history = []
    
    def hook_fn(module, input, output):
        # The TokenPruner returns: Union[torch.Tensor, Tuple[torch.Tensor, dict]]
        x_selected, metadata = output
        if 'selected_indices' in metadata:
            # Store the selected token indices for this layer
            pruned_indices_history.append(metadata['selected_indices'].detach().cpu())
            
    # Attach hook to all TokenPruner instances in the model
    for name, module in model.named_modules():
        if isinstance(module, TokenPruner):
            module.register_forward_hook(hook_fn)
            
    return pruned_indices_history

def visualize_token_pruning(img_vis: Image.Image, pruned_indices_history: list, patch_size: int = 16):
    """
    3. Visualize the spatial token pruning status step-by-step.
    """
    num_stages = len(pruned_indices_history)
    fig, axes = plt.subplots(1, num_stages + 1, figsize=(4 * (num_stages + 1), 4))
    
    # Show Original Cropped Image
    axes[0].imshow(img_vis)
    axes[0].set_title("Original Input")
    axes[0].axis('off')
    
    img_width, img_height = img_vis.size
    num_patches_1d = img_width // patch_size
    num_spatial_patches = num_patches_1d ** 2
    
    # DeiT uses 2 prefix tokens (CLS token + Distillation token)
    prefix_tokens = 2 
    total_initial_tokens = num_spatial_patches + prefix_tokens
    
    # Track global absolute indices of the tokens that survive pruning
    global_alive_indices = torch.arange(total_initial_tokens)
    
    for i, local_kept_indices in enumerate(pruned_indices_history):
        # local_kept_indices is shape [1, current_seq_len]
        local_idx = local_kept_indices[0] 
        
        # Map relative kept indices to absolute global indices
        global_alive_indices = global_alive_indices[local_idx]
        
        # Create a boolean mask where 1.0 = kept token, 0.0 = pruned token
        mask_1d = torch.zeros(total_initial_tokens)
        mask_1d[global_alive_indices] = 1.0
        
        # Discard the prefix tokens (CLS & Dist) and reshape to 2D spatial grid (e.g., 14x14)
        spatial_mask = mask_1d[prefix_tokens:].reshape(num_patches_1d, num_patches_1d).numpy()
        
        # Scale up the 14x14 grid to 224x224 to overlay on the original image
        upscaled_mask = np.kron(spatial_mask, np.ones((patch_size, patch_size)))
        
        # Apply the binary mask to the original image array
        img_np = np.array(img_vis).astype(np.float32) / 255.0
        masked_img = img_np * upscaled_mask[:, :, np.newaxis]
        
        axes[i+1].imshow(masked_img)
        axes[i+1].set_title(f"Stage {i+1} Kept: {len(global_alive_indices)-prefix_tokens}/{num_spatial_patches}")
        axes[i+1].axis('off')

    plt.tight_layout()
    plt.show()

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from compression.compressed_vit import create_compressed_deit_small

IMAGENET_ROOT = '/data/ilsvrc12_raw_torch'  # <-- UPDATE THIS PATH
TARGET_INDEX = 123  # Target sample index
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Init Model ---
# Create the model using your custom factory function
model = create_compressed_deit_small(
    pretrained=False, 
    n_bits=8, 
    reduced_token=40, 
    pruning_layer_indices=[3, 6, 9] 
).to(DEVICE)
model.eval()

# --- Load Data via timm ---
# We use default DeiT configuration (224x224, default mean/std)
default_cfg = {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 
                'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225),
                'crop_pct': 0.875}

try:
    img_tensor, img_vis, label = get_imagenet_sample_timm(
        IMAGENET_ROOT, 
        model_config=default_cfg, 
        target_idx=TARGET_INDEX
    )
    img_tensor = img_tensor.to(DEVICE)
except Exception as e:
    print(f"Failed to load dataset: {e}")
    exit(1)

# --- Run Inference & Collect Metadata ---
# Register the hook to intercept metadata
pruned_indices_history = extract_pruning_metadata(model)

with torch.no_grad():
    output = model(img_tensor)
    print(f"Prediction output shape: {output.shape}")
    
print(f"Captured pruning operations: {len(pruned_indices_history)}")

# --- Visualization ---
if len(pruned_indices_history) > 0:
    visualize_token_pruning(img_vis, pruned_indices_history, patch_size=16)
else:
    print("Warning: No pruning metadata captured.")

NameError: name 'TokenPruner' is not defined